# 01. Exploratory Data Analysis & Clinical Patterns
### Brazilian Medical Appointment No-Shows Dataset (Kaggle)

**Author:** Senior AI/ML Engineer  
**Goal:** Understand demographic drivers, temporal variables, and data quality issues before building predictive models.

## 1. Load Data & Environment Setup

In [ ]:
import os, sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_dataset
from src.data.validate_data import validate_dataset

sns.set_theme(style='whitegrid', palette='muted')
df, data_source = load_dataset()
print(f"Loaded {len(df):,} records from source: {data_source}")
df.head()

## 2. Data Quality & Sanity Audits
Let's check for impossible ages, date anomalies, and missing values.

In [ ]:
report = validate_dataset(df)
print("Validation Status:", report.validation_status)
print("Anomalies detected:", report.anomalies)
print("Missing values per column:", report.missing_values)

**Observations:**
- Exactly 1 record has `Age = -1` (clear data entry typo).
- 5 records have `AppointmentDay < ScheduledDay` (patient scheduled after appointment occurred).
- No missing values in any column.
- We will clean these 6 invalid records in the preprocessing pipeline.

## 3. Target Distribution & Class Imbalance

In [ ]:
counts = df['No-show'].value_counts()
pcts = df['No-show'].value_counts(normalize=True) * 100
print(f"Attended (No): {counts['No']:,} ({pcts['No']:.2f}%)")
print(f"Missed (Yes):   {counts['Yes']:,} ({pcts['Yes']:.2f}%)")

plt.figure(figsize=(6, 4))
plt.bar(['Attended (0)', 'Missed (1)'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
plt.title('Appointment Outcomes (Target Distribution)')
plt.ylabel('Appointments')
plt.show()

**Key Modeling Takeaway:** The dataset has an ~80:20 class imbalance. Standard accuracy is an uninformative metric because a trivial model predicting 'Attended' for all visits gets 79.8% accuracy while catching 0% of no-shows. We must optimize for **Recall, PR-AUC, and F1-Score**.

## 4. Investigating Lead Time (Waiting Days)
How long between the moment a patient schedules an appointment and the actual visit date?

In [ ]:
sched_d = pd.to_datetime(df['ScheduledDay']).dt.floor('D')
appt_d = pd.to_datetime(df['AppointmentDay']).dt.floor('D')
df['Waiting_Days'] = (appt_d - sched_d).dt.days
df['Is_Same_Day'] = (df['Waiting_Days'] == 0).astype(int)
df['No_Show_Num'] = (df['No-show'] == 'Yes').astype(int)

same_day_rate = df[df['Is_Same_Day'] == 1]['No_Show_Num'].mean() * 100
advance_rate = df[df['Is_Same_Day'] == 0]['No_Show_Num'].mean() * 100

print(f"Same-Day Appointment No-Show Rate:   {same_day_rate:.2f}%")
print(f"Advance-Booked Appointment No-Show:  {advance_rate:.2f}%")

**Critical Insight:** Same-day appointments miss only **4.65%** of the time. Advance-booked appointments miss **28.5%** of the time. Lead time is by far the single strongest predictive feature.

## 5. Unpacking the SMS Reminder Paradox (Simpson's Paradox)
Let's see why raw statistics suggest SMS reminders increase no-shows.

In [ ]:
sms_rates = df.groupby('SMS_received')['No_Show_Num'].agg(['mean', 'count'])
sms_rates['mean'] *= 100
sms_rates.columns = ['No-Show Rate (%)', 'Count']
print(sms_rates)

# Breakdown by same-day vs advance booking:
pivot = df.pivot_table(index='SMS_received', columns='Is_Same_Day', values='No_Show_Num', aggfunc='mean') * 100
pivot.columns = ['Advance Booking (>0d)', 'Same-Day Booking (0d)']
print("\nControlled for Lead Time:")
print(pivot)

**Conclusion on SMS:** Notice that for same-day appointments, `SMS_received` is strictly 0 (clinics never text on the same morning). Because same-day appointments have near-zero no-shows, they drag down the overall average for the 'No SMS' group. Once we look only at advance bookings, SMS does not cause no-shows. Lead time is the real confounder.